# SEIS 606: Vibe Coding
## Homework 2, Image Generation for App Specs
Dante Razo, razo3843@stthomas.edu, FA26

I've been using this GPU-accelerated notebook template since I first started at UST. It's something that I carry from class to class.

## GPU-Accelerated Environment Configuration

In [1]:
import torch

# validate CUDA setup
print("Torch CUDA Available? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Torch CUDA Version:", torch.version.cuda)
    print("Torch cuDNN Version:", torch.backends.cudnn.version())

    # print GPU information
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:", torch.cuda.get_device_name(device=i))

    # check NVIDIA driver
    !echo && nvidia-smi

# set device type
device: str = "cuda" if torch.cuda.is_available() else "cpu"

Torch CUDA Available?  True
Torch CUDA Version: 13.0
Torch cuDNN Version: 92400

GPU 0: NVIDIA GeForce RTX 5090

Fri Sep 25 02:37:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.08              KMD Version: 616.92        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
| 30%   35C    P0             68W /  460W |    2710MiB /  32607MiB |      5%      Default |
|                          

In [2]:
import gc


def free_vram() -> None:
    gc.collect()
    torch.cuda.empty_cache()


# free now, though it should be empty with a fresh kernel
free_vram()

In [3]:
import os
from pathlib import Path

# create cache location
hf_home: Path = Path("/cache/huggingface")
hf_home.mkdir(parents=True, exist_ok=True)

# set environment variables for huggingface / transformers
os.environ["HF_HOME"] = str(object=hf_home)

In [4]:
from dotenv import load_dotenv

# load environment, including HF token
load_dotenv()

False

In [5]:
# validate environment variables with assertions
assert hf_home.exists()
assert os.environ["HF_HOME"] == str(object=hf_home)

## Loading the Image Generation Model
For simplicity and compatibility with a 32GB RTX 5090, this notebook uses **SDXL 1.0** in native fp16 weights from Hugging Face (no runtime quantization).

In [ ]:
from diffusers.models.transformers.transformer_qwenimage import QwenImageTransformer2DModel
from diffusers.pipelines.qwenimage.pipeline_qwenimage import QwenImagePipeline
from diffusers.quantizers.quantization_config import BitsAndBytesConfig
from torch import bfloat16
from transformers import (
    BitsAndBytesConfig as TransformersBitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

model_id: str = "Qwen/Qwen-Image"

# define object for and quantize transformer component of model
transformer: QwenImageTransformer2DModel = QwenImageTransformer2DModel.from_pretrained(
    pretrained_model_name_or_path=model_id,
    subfolder="transformer",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    dtype=bfloat16,
)

# define object for and quantize text encoder component of model
text_encoder: Qwen2_5_VLForConditionalGeneration = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=model_id,
    subfolder="text_encoder",
    quantization_config=TransformersBitsAndBytesConfig(load_in_8bit=True),
    dtype=bfloat16,
)

# define full pipeline
pipeline: QwenImagePipeline = QwenImagePipeline.from_pretrained(
    pretrained_model_name_or_path=model_id,
    transformer=transformer,
    text_encoder=text_encoder,
    dtype=bfloat16,
).to(device)

# improve memory efficiency
pipeline.enable_attention_slicing()

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
from torch import inference_mode


def generate_image(prompt: str, steps: int = 30, guidance: float = 7.0, save_path: str = "") -> None:
    """Wrapper function for generation + preview + persisting to disk."""
    with inference_mode():
        image = pipeline(
            prompt=prompt,
            negative_prompt=" ",
            num_inference_steps=steps,
            true_cfg_scale=guidance,
        ).images[0]

    # display generated image
    display(image)

    # optionally persist the generated image to disk
    image.save(save_path) if save_path else None

In [ ]:
prompt: str = """
A clean UI mockup for a homelab dashboard in the style of early 2000s websites. Use interesting shapes, vibrant colors, and a playful layout. Invoke the Y2K, 2Advanced, and/or Metalheart aesthetic(s). Refer to Final Fantasy 13 for an anachronistic example.

Ensure text is legible and large enough to see. Avoid overlapping elements and maintain a clear hierarchy. Keep text artifacts to a minimum.

Create a summary panel for server information. Include key metrics like CPU usage, memory usage, zpool status, and network activity. Do that for five servers: Kveikur, Kex, Hoppípolla, Sæglópur, and Cerulean.
"""

# take 1
generate_image(prompt=prompt, steps=35, guidance=5.5)

In [ ]:
# take 2
generate_image(prompt=prompt, steps=30, guidance=8.0)

In [ ]:
# take 3
generate_image(prompt=prompt, steps=25, guidance=6.5)